In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import random
import os
from tqdm import tqdm # Biblioteca para barra de progresso

In [2]:
def plot_beat_segment(segment_df, title="Segmento de Batimento Cardíaco"):
    """Função auxiliar para plotar um segmento de batimento de um DataFrame."""
    if segment_df is None or segment_df.empty:
        print("DataFrame vazio. Nada para plotar.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(segment_df["sample #"], segment_df["amplitude"], label="Sinal")
    plt.title(title)
    plt.xlabel("Número da Amostra")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [3]:
def augment_scale_and_jitter(segment_df, scale_range=(0.9, 1.1), sigma_factor=0.02):
    """
    Aplica uma sequência de aumentos de dados: primeiro AmplitudeScale e depois Jitter.

    Parâmetros:
    - segment_df (pd.DataFrame): O DataFrame do segmento original.
    - scale_range (tuple): O intervalo para o AmplitudeScale.
    - sigma_factor (float): O fator de sigma para o Jitter.

    Retorna:
    - pd.DataFrame: O DataFrame com as duas transformações aplicadas.
    """
    #  Aplicar a escala de amplitude
    scaled_df = augment_amplitude_scale(segment_df, scale_range=scale_range)
    
    # Aplicar o jitter no resultado da escala
    final_augmented_df = augment_jitter(scaled_df, sigma_factor=sigma_factor)
    
    return final_augmented_df

In [4]:
def save_augmented_segment(segment_df, output_dir, file_name):
    """
    Salva um único segmento em CSV, garantindo o formato correto das colunas.
    """
    # Garante que o DataFrame tenha as colunas no formato original para salvar
    augmented_df = segment_df.copy()
    
    if "amplitude" in augmented_df.columns:
        augmented_df["channel_0"] = augmented_df["amplitude"]
    
    # Garante que as colunas essenciais existam
    if "type" not in augmented_df.columns: augmented_df['type'] = 'unknown'
    if "sample #" not in augmented_df.columns: augmented_df['sample #'] = np.arange(len(augmented_df))

    # Seleciona e ordena as colunas
    augmented_df = augmented_df[["channel_0", "sample #", "type"]]
    
    # Constrói o caminho completo do arquivo e salva
    full_path = os.path.join(output_dir, file_name)
    augmented_df.to_csv(full_path, index=False)

In [5]:
def augment_amplitude_scale(segment_df, scale_range=(0.8, 1.2)):
    """
    Aplica uma escala na amplitude do sinal, multiplicando todos os valores
    por um fator aleatório dentro do intervalo especificado.

    Parâmetros:
    - segment_df (pd.DataFrame): DataFrame com a coluna 'amplitude'.
    - scale_range (tuple): Tupla (min, max) para o fator de escala aleatório.

    Retorna:
    - pd.DataFrame: O DataFrame com a amplitude escalada.
    """
    augmented_df = segment_df.copy()
    
    # Sorteia um fator de escala aleatório do intervalo
    scale_factor = random.uniform(scale_range[0], scale_range[1])
    
    # Aplica a multiplicação na coluna de amplitude
    augmented_df["amplitude"] = segment_df["amplitude"] * scale_factor
    
    return augmented_df

In [6]:
# Versão ajustada da augment_jitter
def augment_jitter(segment_df, sigma_factor=0.02, variable_sigma=True, seed=None):
    # Removi save_path e number dos argumentos
    rng = np.random.default_rng(seed)
    augmented_df = segment_df.copy()
    base_sigma = sigma_factor * np.std(segment_df["amplitude"])
    
    if variable_sigma:
        sigma_series = base_sigma * rng.lognormal(mean=0, sigma=0.25, size=len(segment_df))
    else:
        sigma_series = np.full(len(segment_df), base_sigma)
        
    noise = rng.normal(0, sigma_series)
    augmented_df["amplitude"] += noise
    
    # A linha que salvava o arquivo foi removida
    return augmented_df

In [7]:
def get_beat_interval_robust_optimized(df, all_peaks, target_rows, nth=0, channel="channel_0"):
    """
    Versão otimizada que recebe um DataFrame e picos pré-calculados para extrair
    o intervalo R-R de um batimento específico.
    """
    if nth >= len(target_rows):
        return None

    center_sample = int(target_rows.iloc[nth]["sample #"])

    # Encontra o índice do pico mais próximo da anotação
    center_peak_index = np.argmin(np.abs(all_peaks - center_sample))
    
    # Verificação de borda
    if center_peak_index == 0 or center_peak_index >= len(all_peaks) - 1:
        return None

    # Pega os picos vizinhos
    start_peak = all_peaks[center_peak_index - 1]
    end_peak = all_peaks[center_peak_index + 1]
    
    # Extrai o segmento
    mask = (df["sample #"] >= start_peak) & (df["sample #"] <= end_peak)
    segment_df = df.loc[mask].copy()

    # Renomeia a coluna para o padrão "amplitude"
    if channel in segment_df.columns:
        segment_df.rename(columns={channel: "amplitude"}, inplace=True)
    
    return segment_df

In [8]:
# --- Execução Otimizada para Múltiplos Arquivos ---

if __name__ == '__main__':
    # --- Parâmetros ---
    csv_file = "mitbih_all_records_renumerada.csv"
    output_dir = "data_amplitude_scale_jitter_multiple_files/"
    target_type = "R"
    num_augmentations = 7200
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25
    
    # --- Pré-cálculo ---
    os.makedirs(output_dir, exist_ok=True)
    
    print("1/3 - Carregando arquivo CSV...")
    df_main = pd.read_csv(csv_file)
    
    print(f"2/3 - Encontrando anotações '{target_type}'...")
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        num_augmentations = len(target_rows)
        print(f"Aviso: Reduzindo para {num_augmentations} aumentos, pois é o total de batimentos encontrados.")

    print("3/3 - Detectando picos R...")
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=peak_distance, height=peak_height)
    
    # --- Loop de Aumento e Salvamento ---
    print(f"Gerando e salvando {num_augmentations} arquivos individuais...")
    for i in tqdm(range(num_augmentations), desc="Gerando Arquivos"):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # Chama a nova função combinada
            augmented_df = augment_scale_and_jitter(segment_df, scale_range=(0.9, 1.1), sigma_factor=0.015)
            
            # Define o novo nome do arquivo
            file_name = f"{target_type}_beat_{i}_aug_scale_jitter.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
            
    print(f"Processo concluído! {num_augmentations} arquivos salvos em '{output_dir}'.")

1/3 - Carregando arquivo CSV...
2/3 - Encontrando anotações 'R'...
3/3 - Detectando picos R...
Gerando e salvando 7200 arquivos individuais...


Gerando Arquivos: 100%|█████████████████████████████████████████████████████████████| 7200/7200 [08:35<00:00, 13.98it/s]

Processo concluído! 7200 arquivos salvos em 'data_amplitude_scale_jitter_multiple_files/'.


In [9]:
# --- Execução Otimizada para Arquivo Único ---

if __name__ == '__main__':
    # --- Parâmetros ---
    csv_file = "mitbih_all_records_renumerada.csv"
    output_file = "augmented_beats_scale_jitter_single_file_L.csv"
    target_type = "L"
    num_augmentations = 8000
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25
    
    # --- Pré-cálculo ---
    print("1/3 - Carregando arquivo CSV...")
    df_main = pd.read_csv(csv_file)
    
    print(f"2/3 - Encontrando anotações '{target_type}'...")
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        num_augmentations = len(target_rows)
        print(f"Aviso: Reduzindo para {num_augmentations} aumentos, pois é o total de batimentos encontrados.")

    print("3/3 - Detectando picos R...")
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=peak_distance, height=peak_height)
    
    # --- Loop de Aumento e Coleta ---
    all_augmented_segments = []
    print(f"Gerando {num_augmentations} aumentos de dados...")
    for i in tqdm(range(num_augmentations), desc="Processando Batimentos"):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # Chama a nova função combinada
            augmented_df = augment_scale_and_jitter(segment_df, scale_range=(0.9, 1.1), sigma_factor=0.015)
            
            # Adiciona um ID para identificar o batimento no arquivo final
            augmented_df['beat_id'] = i
            
            all_augmented_segments.append(augmented_df)
            
    # --- Consolidação e Salvamento Único ---
    if all_augmented_segments:
        print("Consolidando todos os segmentos...")
        # Recria as colunas originais antes de salvar
        final_df = pd.concat(all_augmented_segments, ignore_index=True)
        final_df.rename(columns={'amplitude': 'channel_0'}, inplace=True)
        
        print(f"Salvando em um único arquivo: {output_file}")
        final_df.to_csv(output_file, index=False)
        print("Processo concluído!")
    else:
        print("Nenhum segmento foi gerado.")

1/3 - Carregando arquivo CSV...
2/3 - Encontrando anotações 'L'...
3/3 - Detectando picos R...
Gerando 8000 aumentos de dados...


Processando Batimentos: 100%|███████████████████████████████████████████████████████| 8000/8000 [09:39<00:00, 13.79it/s]


Consolidando todos os segmentos...
Salvando em um único arquivo: augmented_beats_scale_jitter_single_file_L.csv
Processo concluído!
